In [ ]:
# IMPORTS
import os, random, warnings, csv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image

from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
# DEVICE
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

In [ ]:
# SEED
def seed_everything(seed: int = 7):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(7)
print('Seed = 7')

In [ ]:
# OUTPUT FOLDER STRUCTURE

RESULTS_DIR = 'results_split_fraction_study'
DIR_B1  = os.path.join(RESULTS_DIR, 'block1_overall_sweep')
DIR_B2  = os.path.join(RESULTS_DIR, 'block2_perrock_analysis')
DIR_FIN = os.path.join(RESULTS_DIR, 'final_comparison')

for d in [RESULTS_DIR, DIR_B1, DIR_B2, DIR_FIN]:
    os.makedirs(d, exist_ok=True)

_saved_files = []   

def save_fig(fig, folder: str, filename: str, description: str, dpi: int = 150):
    """Save fig to folder/filename and register it in the results index."""
    path = os.path.join(folder, filename)
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    _saved_files.append((path, description))
    print(f'[SAVED] {path}')
    print(f'        -> {description}')

print('Output directories created.')
print(f'  {DIR_B1}')
print(f'  {DIR_B2}')
print(f'  {DIR_FIN}')

In [ ]:
# EXPERIMENT CONFIG
BASE_PATH = os.path.expanduser('~/Desktop/george/rocks_spectral_224')

LR           = 1e-4
WEIGHT_DECAY = 1e-4
BATCH_SIZE   = 64
EPOCHS       = 15    

DATA_FRACTIONS = [0.20, 0.40, 0.60, 0.80]

TEST_SIZES = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

n_runs = len(DATA_FRACTIONS) * len(TEST_SIZES)
print(f'Data fractions : {[f"{int(f*100)}%" for f in DATA_FRACTIONS]}')
print(f'Test-set sizes : {[f"{int(t*100)}%" for t in TEST_SIZES]}')
print(f'Runs per dataset : {n_runs}')
print(f'Grand total      : {n_runs * 2}  (1.83 Hz + 5.10 Hz)')

In [ ]:
# TRANSFORMS
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std),
])
print('Transforms ready.')

In [ ]:
# DATA LOADING
VALID_EXT = ('.jpg', '.jpeg', '.bmp', '.png')

folders_183 = {
    'S10Granite':           os.path.join(BASE_PATH, 'S10Granite_1-83Hz_Spectral'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'Holstein_Sandstone_1-83Hz_Spectral'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'Leitendorf_Limestone_1-83Hz_Spectral'),
}
folders_510 = {
    'S10Granite':           os.path.join(BASE_PATH, 'S10Granite_5-10Hz_Spectral'),
    'Holstein_Sandstone':   os.path.join(BASE_PATH, 'Holstein_Sandstone_5-10Hz_Spectral'),
    'Leitendorf_Limestone': os.path.join(BASE_PATH, 'Leitendorf_Limestone_5-10Hz_Spectral'),
}

def collect_image_paths(folders_dict):
    image_paths, labels = [], []
    class_names = list(folders_dict.keys())
    for label, (rock_name, folder) in enumerate(folders_dict.items()):
        files = sorted([f for f in os.listdir(folder) if f.lower().endswith(VALID_EXT)])
        for fname in files:
            image_paths.append(os.path.join(folder, fname))
            labels.append(label)
        print(f'  {rock_name}: {len(files)} images')
    return image_paths, labels, class_names

print('\n=== 1.83 Hz ===')
paths_183, labels_183, class_names = collect_image_paths(folders_183)
print(f'  TOTAL: {len(paths_183)}')

print('\n=== 5.10 Hz ===')
paths_510, labels_510, _ = collect_image_paths(folders_510)
print(f'  TOTAL: {len(paths_510)}')

short_names = [c.split('_')[0] for c in class_names]  
print(f'\nClasses: {class_names}')

In [ ]:
# DATASET CLASS
class SpectralImageDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels      = labels
        self.transform   = transform

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        img = Image.open(self.image_paths[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

print('SpectralImageDataset ready.')

In [ ]:
# MODEL BUILDER (ResNet-18 only)
def build_resnet18(n_classes: int) -> nn.Module:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, n_classes))
    return model

_m = build_resnet18(3)
print(f'ResNet-18 | Total params: {sum(p.numel() for p in _m.parameters()):,} | '
      f'Trainable: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}')
del _m

In [ ]:
# CORE TRAIN + EVAL FUNCTION
def train_single_run(
    paths_all, labels_all, class_names,
    data_fraction: float,
    test_size: float,
    epochs: int = EPOCHS,
    seed: int = 7
) -> dict:
    """
    1. Stratified sub-sample data_fraction of the full dataset.
    2. Stratified train/test split using test_size.
    3. Train ResNet-18.
    4. Return metrics dict or None if the split is degenerate.
    """
    seed_everything(seed)
    n_classes  = len(class_names)
    labels_arr = np.array(labels_all)
    all_idx    = np.arange(len(paths_all))

    if data_fraction >= 1.0:
        subset_idx = all_idx
    else:
        sss = StratifiedShuffleSplit(n_splits=1, test_size=data_fraction, random_state=seed)
        _, subset_idx = next(sss.split(all_idx, labels_arr))

    paths_sub  = [paths_all[i]  for i in subset_idx]
    labels_sub = [labels_all[i] for i in subset_idx]

    if min(labels_sub.count(c) for c in range(n_classes)) < 2:
        return None

    paths_train, paths_test, labels_train, labels_test = train_test_split(
        paths_sub, labels_sub, test_size=test_size, stratify=labels_sub, random_state=seed
    )
    if len(paths_train) < 3 or len(paths_test) < 3:
        return None

    nw        = min(4, os.cpu_count() or 1)
    eff_batch = min(BATCH_SIZE, len(paths_train))
    pin_mem   = (device.type == 'cuda')  # pin_memory only valid on CUDA
    train_ds  = SpectralImageDataset(paths_train, labels_train, train_transform)
    test_ds   = SpectralImageDataset(paths_test,  labels_test,  test_transform)
    train_ldr = DataLoader(train_ds, eff_batch, shuffle=True,
                            num_workers=nw, pin_memory=pin_mem, persistent_workers=(nw > 0))
    test_ldr  = DataLoader(test_ds,  eff_batch, shuffle=False,
                            num_workers=nw, pin_memory=pin_mem, persistent_workers=(nw > 0))

    use_amp   = (device.type == 'cuda')
    model     = build_resnet18(n_classes).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=2)
    warmup    = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=2)
    criterion = nn.CrossEntropyLoss()
    scaler    = torch.amp.GradScaler('cuda') if use_amp else None

    train_accs, test_accs = [], []
    best_acc  = -1.0   
    best_true, best_pred = [], []

    for epoch in range(epochs):
        model.train()
        tr_buf = []
        for Xb, yb in train_ldr:
            Xb, yb = Xb.to(device), yb.to(device)
            optimizer.zero_grad()
            if use_amp:
                with torch.amp.autocast('cuda'):
                    out  = model(Xb)
                    loss = criterion(out, yb)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                scaler.step(optimizer)
                scaler.update()
            else:
                out  = model(Xb)
                loss = criterion(out, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
                optimizer.step()
            tr_buf.append((out.argmax(1) == yb).float().mean().item())
        train_accs.append(float(np.mean(tr_buf)))

        model.eval()
        te_buf, ep_true, ep_pred = [], [], []
        with torch.no_grad():
            for Xb, yb in test_ldr:
                Xb, yb = Xb.to(device), yb.to(device)
                preds  = model(Xb).argmax(1)
                te_buf.append((preds == yb).float().mean().item())
                ep_true.extend(yb.cpu().numpy().tolist())
                ep_pred.extend(preds.cpu().numpy().tolist())

        te_acc = float(np.mean(te_buf))
        test_accs.append(te_acc)
        if epoch < 2: warmup.step()
        else:         scheduler.step(te_acc)

        if te_acc > best_acc:
            best_acc  = te_acc
            best_true = ep_true[:]
            best_pred = ep_pred[:]

    cm     = confusion_matrix(best_true, best_pred, labels=list(range(n_classes)))
    report = classification_report(
        best_true, best_pred, target_names=class_names, output_dict=True, zero_division=0)

    del model, optimizer, criterion
    if scaler is not None:
        del scaler
    torch.cuda.empty_cache()

    return {
        'best_acc'         : best_acc,
        'per_class_recall' : {c: report[c]['recall']   for c in class_names},
        'per_class_f1'     : {c: report[c]['f1-score'] for c in class_names},
        'cm'               : cm,
        'train_accs'       : train_accs,
        'test_accs'        : test_accs,
        'n_train'          : len(paths_train),
        'n_test'           : len(paths_test),
    }

print('train_single_run() defined.')

In [ ]:
# FULL GRID SWEEP

datasets = {
    '1.83 Hz': (paths_183, labels_183),
    '5.10 Hz': (paths_510, labels_510),
}
sweep_results = {tag: {} for tag in datasets}
total_runs = len(datasets) * len(DATA_FRACTIONS) * len(TEST_SIZES)
run_n = 0

for speed_tag, (paths_all, labels_all) in datasets.items():
    print(f'\n{"="*60}')
    print(f' {speed_tag}  ({len(paths_all)} images)')
    print(f'{"="*60}')
    for frac in DATA_FRACTIONS:
        for ts in TEST_SIZES:
            run_n += 1
            print(f'  [{run_n:>3}/{total_runs}]  data={int(frac*100):>3}%  test={int(ts*100):>3}%',
                  end=' ... ', flush=True)
            r = train_single_run(
                paths_all, labels_all, class_names,
                data_fraction=frac, test_size=ts, epochs=EPOCHS
            )
            sweep_results[speed_tag][(frac, ts)] = r
            if r is None:
                print('SKIPPED (degenerate split)')
            else:
                print(f'acc={r["best_acc"]*100:.2f}%  '
                      f'(n_train={r["n_train"]}, n_test={r["n_test"]})')

print('\n Sweep complete.')

In [ ]:
# MATRIX BUILDER HELPERS
def acc_matrix(rd):
    """2-D array (rows=DATA_FRACTIONS, cols=TEST_SIZES) of best accuracy in %."""
    mat = np.full((len(DATA_FRACTIONS), len(TEST_SIZES)), np.nan)
    for fi, frac in enumerate(DATA_FRACTIONS):
        for ti, ts in enumerate(TEST_SIZES):
            r = rd.get((frac, ts))
            if r: mat[fi, ti] = r['best_acc'] * 100
    return mat

def recall_matrix(rd, cls_name):
    """Same shape; values = per-class recall %."""
    mat = np.full((len(DATA_FRACTIONS), len(TEST_SIZES)), np.nan)
    for fi, frac in enumerate(DATA_FRACTIONS):
        for ti, ts in enumerate(TEST_SIZES):
            r = rd.get((frac, ts))
            if r: mat[fi, ti] = r['per_class_recall'][cls_name] * 100
    return mat

def valid_dict(rd):
    return {k: v for k, v in rd.items() if v is not None}

xl = [f'{int(ts*100)}%'   for ts   in TEST_SIZES]
yl = [f'{int(f*100)}%'    for f    in DATA_FRACTIONS]

print('Matrix helpers ready.')

In [ ]:
# B1-01  Overall accuracy heatmap

fig, axes = plt.subplots(1, 2, figsize=(20, 6))
fig.suptitle(
    'B1-01 | Overall Test Accuracy Heatmap -- ResNet-18\n'
    'Row = fraction of total images used before splitting  |  Col = fraction of those images used as test set\n'
    'Cell = best test accuracy (%) reached during training  |  Green = high, Red = low',
    fontsize=12, fontweight='bold'
)
for ax, tag in zip(axes, ['1.83 Hz', '5.10 Hz']):
    sns.heatmap(acc_matrix(sweep_results[tag]), ax=ax,
                xticklabels=xl, yticklabels=yl,
                annot=True, fmt='.1f', cmap='RdYlGn', vmin=33, vmax=100,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'Test Accuracy (%)'})
    ax.set_title(f'Belt speed: {tag}', fontsize=11)
    ax.set_xlabel('Test-Set Size (% of sub-sampled data)')
    ax.set_ylabel('Data Fraction (% of total images used)')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
save_fig(fig, DIR_B1,
    'B1-01_overall_accuracy_heatmap__datafrac_vs_testsplit.png',
    'Block 1 | Heatmap of overall test accuracy (%) for every (data-fraction x test-split) combo -- both belt speeds side by side')
plt.show()

In [ ]:
# B1-02  Accuracy vs. test-split (one line per data fraction)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
fig.suptitle(
    'B1-02 | Overall Accuracy vs. Test-Split Ratio -- one line per data-fraction level\n'
    'X-axis = % of sub-sampled data reserved as test set  |  Lines drop when training data becomes scarce',
    fontsize=11, fontweight='bold'
)
lc = plt.cm.viridis(np.linspace(0.1, 0.9, len(DATA_FRACTIONS)))
for ax, tag in zip(axes, ['1.83 Hz', '5.10 Hz']):
    mat = acc_matrix(sweep_results[tag])
    for fi, (frac, col) in enumerate(zip(DATA_FRACTIONS, lc)):
        row = mat[fi]; valid = ~np.isnan(row)
        ax.plot(np.array(TEST_SIZES)[valid]*100, row[valid],
                marker='o', color=col, label=f'{int(frac*100)}% data')
    ax.set_title(f'Belt speed: {tag}', fontsize=11)
    ax.set_xlabel('Test-Set Size (% of sub-sampled data)')
    ax.set_ylabel('Best Test Accuracy (%)')
    ax.set_xticks([int(t*100) for t in TEST_SIZES])
    ax.set_ylim(30, 103)
    ax.axhline(33.3, color='grey', ls='--', lw=0.8, alpha=0.5, label='Random (3 classes)')
    ax.legend(title='Data fraction', fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, DIR_B1,
    'B1-02_overall_accuracy_vs_testsplit__one_line_per_datafraction.png',
    'Block 1 | Accuracy curves vs. test-split ratio -- one line per data-fraction level for both datasets')
plt.show()

In [ ]:
# B1-03  Accuracy vs. data fraction (one line per test-split)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
fig.suptitle(
    'B1-03 | Overall Accuracy vs. Data Fraction -- one line per test-split size\n'
    'X-axis = % of total images used before the train/test split  |  Rising lines = model benefits from more data',
    fontsize=11, fontweight='bold'
)
lc2 = plt.cm.plasma(np.linspace(0.1, 0.9, len(TEST_SIZES)))
mk2 = ['o','s','^','D','v','<','>','p','*']
for ax, tag in zip(axes, ['1.83 Hz', '5.10 Hz']):
    mat = acc_matrix(sweep_results[tag])
    for ti, (ts, col, mk) in enumerate(zip(TEST_SIZES, lc2, mk2)):
        col_d = mat[:,ti]; valid = ~np.isnan(col_d)
        ax.plot(np.array(DATA_FRACTIONS)[valid]*100, col_d[valid],
                marker=mk, color=col, label=f'{int(ts*100)}% test')
    ax.set_title(f'Belt speed: {tag}', fontsize=11)
    ax.set_xlabel('Data Fraction (% of total images)')
    ax.set_ylabel('Best Test Accuracy (%)')
    ax.set_xticks([int(f*100) for f in DATA_FRACTIONS])
    ax.set_ylim(30, 103)
    ax.axhline(33.3, color='grey', ls='--', lw=0.8, alpha=0.5)
    ax.legend(title='Test-split size', fontsize=8)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, DIR_B1,
    'B1-03_overall_accuracy_vs_datafraction__one_line_per_testsplit.png',
    'Block 1 | Accuracy curves vs. data fraction -- one line per test-split size for both datasets')
plt.show()

In [ ]:
# B1-04  Training history: BEST and WORST configuration per dataset

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle(
    'B1-04 | Training History -- Best vs. Worst Configuration per Dataset\n'
    'Blue = training accuracy per epoch  |  Red = test accuracy per epoch  |  Green dashed = best test accuracy reached\n'
    'Row = dataset  |  Left col = BEST config  |  Right col = WORST config',
    fontsize=11, fontweight='bold'
)
for ri, tag in enumerate(['1.83 Hz', '5.10 Hz']):
    vd = valid_dict(sweep_results[tag])
    bk = max(vd, key=lambda k: vd[k]['best_acc'])
    wk = min(vd, key=lambda k: vd[k]['best_acc'])
    for ci, (key, lbl) in enumerate([(bk,'BEST'),(wk,'WORST')]):
        r = vd[key]; ax = axes[ri][ci]
        frac, ts = key
        ep = range(1, len(r['train_accs'])+1)
        ax.plot(ep, np.array(r['train_accs'])*100, 'b-o', ms=4, label='Train acc')
        ax.plot(ep, np.array(r['test_accs'])*100,  'r-s', ms=4, label='Test acc')
        ax.axhline(r['best_acc']*100, color='green', ls='--', lw=1.2,
                   label=f'Best test: {r["best_acc"]*100:.1f}%')
        ax.set_title(
            f'[{lbl}] {tag}\n'
            f'data-fraction={int(frac*100)}%  test-split={int(ts*100)}%  '
            f'n_train={r["n_train"]}  n_test={r["n_test"]}', fontsize=9)
        ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
        ax.set_ylim(0, 105); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, DIR_B1,
    'B1-04_training_history__best_and_worst_config_per_dataset.png',
    'Block 1 | Epoch-by-epoch accuracy curves for the best and worst (data-fraction, test-split) config on each dataset')
plt.show()

In [ ]:
# B1-05  Confusion matrix: BEST configuration per dataset

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'B1-05 | Confusion Matrix -- Best Configuration per Dataset\n'
    'Diagonal = correct predictions  |  Off-diagonal = misclassifications  |  Values = raw image counts',
    fontsize=11, fontweight='bold'
)
for ax, tag in zip(axes, ['1.83 Hz', '5.10 Hz']):
    vd = valid_dict(sweep_results[tag])
    bk = max(vd, key=lambda k: vd[k]['best_acc'])
    r = vd[bk]; frac, ts = bk
    sns.heatmap(r['cm'], ax=ax, annot=True, fmt='d', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names)
    ax.set_title(f'Belt speed: {tag}\ndata={int(frac*100)}%  test-split={int(ts*100)}%  acc={r["best_acc"]*100:.1f}%', fontsize=10)
    ax.set_ylabel('Actual class'); ax.set_xlabel('Predicted class')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
save_fig(fig, DIR_B1,
    'B1-05_confusion_matrix__best_config_per_dataset.png',
    'Block 1 | Confusion matrix (raw counts) for the best (data-fraction, test-split) config on each dataset')
plt.show()

In [ ]:
# B1-06  Confusion matrix: WORST configuration per dataset

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle(
    'B1-06 | Confusion Matrix -- Worst Configuration per Dataset\n'
    'Diagonal = correct predictions  |  Off-diagonal = misclassifications  |  Values = raw image counts\n'
    'Shows which rock pairs are most confused under low-data / minimal-training conditions',
    fontsize=11, fontweight='bold'
)
for ax, tag in zip(axes, ['1.83 Hz', '5.10 Hz']):
    vd = valid_dict(sweep_results[tag])
    wk = min(vd, key=lambda k: vd[k]['best_acc'])
    r = vd[wk]; frac, ts = wk
    sns.heatmap(r['cm'], ax=ax, annot=True, fmt='d', cmap='Reds',
                xticklabels=short_names, yticklabels=short_names)
    ax.set_title(f'Belt speed: {tag}\ndata={int(frac*100)}%  test-split={int(ts*100)}%  acc={r["best_acc"]*100:.1f}%', fontsize=10)
    ax.set_ylabel('Actual class'); ax.set_xlabel('Predicted class')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
save_fig(fig, DIR_B1,
    'B1-06_confusion_matrix__worst_config_per_dataset.png',
    'Block 1 | Confusion matrix (raw counts) for the worst (data-fraction, test-split) config on each dataset')
plt.show()

In [ ]:
# B1-07  Confusion matrix grid: fixed test-split, varying data fraction

FIXED_TS = 0.30
assert FIXED_TS in TEST_SIZES, (
    f'FIXED_TS={FIXED_TS} is not in TEST_SIZES={TEST_SIZES}. '
    f'Either add it to TEST_SIZES or change FIXED_TS to one of {TEST_SIZES}.'
)
for tag in ['1.83 Hz', '5.10 Hz']:
    tag_fs = tag.replace(' ','').replace('.','')
    fig, axes = plt.subplots(1, len(DATA_FRACTIONS), figsize=(6*len(DATA_FRACTIONS), 5))
    fig.suptitle(
        f'B1-07 | Confusion Matrix Grid -- Fixed test-split={int(FIXED_TS*100)}%, Varying data fraction  [{tag}]\n'
        f'Panels left to right = {int(DATA_FRACTIONS[0]*100)}% to {int(DATA_FRACTIONS[-1]*100)}% of total images used\n'
        f'Shows how the classification pattern improves as more images are included',
        fontsize=11, fontweight='bold'
    )
    for ax, frac in zip(axes, DATA_FRACTIONS):
        r = sweep_results[tag].get((frac, FIXED_TS))
        if r is None:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{int(frac*100)}% data\nN/A'); continue
        sns.heatmap(r['cm'], ax=ax, annot=True, fmt='d', cmap='Blues',
                    xticklabels=short_names, yticklabels=short_names, cbar=False)
        ax.set_title(f'{int(frac*100)}% data\nacc={r["best_acc"]*100:.1f}%', fontsize=10)
        ax.set_ylabel('Actual' if frac == DATA_FRACTIONS[0] else '')
        ax.set_xlabel('Predicted'); ax.tick_params(axis='x', rotation=25)
    plt.tight_layout()
    save_fig(fig, DIR_B1,
        f'B1-07_confusion_matrix_grid__fixed_testsplit{int(FIXED_TS*100)}pct__varying_datafrac__{tag_fs}.png',
        f'Block 1 | CM grid: test-split fixed at {int(FIXED_TS*100)}%, data fraction varies {int(DATA_FRACTIONS[0]*100)}% to {int(DATA_FRACTIONS[-1]*100)}%  [{tag}]')
    plt.show()

In [ ]:
# B2-01  Per-rock recall heatmaps  (3 rocks x 2 datasets)

fig, axes = plt.subplots(len(class_names), 2, figsize=(20, 5*len(class_names)))
fig.suptitle(
    'B2-01 | Per-Rock Recall Heatmaps (%)\n'
    'Each sub-plot = one rock type on one belt speed\n'
    'Row = data fraction  |  Col = test-set size  |  Value = recall % (TP / all actual positives for that rock)',
    fontsize=12, fontweight='bold'
)
for ci, cls in enumerate(class_names):
    for si, tag in enumerate(['1.83 Hz', '5.10 Hz']):
        ax = axes[ci][si]
        sns.heatmap(recall_matrix(sweep_results[tag], cls), ax=ax,
                    xticklabels=xl, yticklabels=yl,
                    annot=True, fmt='.1f', cmap='RdYlGn', vmin=0, vmax=100,
                    linewidths=0.5, linecolor='white',
                    cbar_kws={'label': 'Recall (%)'})
        ax.set_title(f'{cls.replace("_"," ")} -- {tag}', fontsize=11)
        ax.set_xlabel('Test-Set Size'); ax.set_ylabel('Data Fraction')
        ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
save_fig(fig, DIR_B2,
    'B2-01_perrock_recall_heatmaps__all_rocks_both_datasets.png',
    'Block 2 | Per-rock recall (%) heatmaps for all 3 rocks across the full data-fraction x test-split grid -- both datasets')
plt.show()

In [ ]:
# B2-02  Per-rock recall vs. test-split  (one line per data fraction)

lc = plt.cm.viridis(np.linspace(0.1, 0.9, len(DATA_FRACTIONS)))
mk = ['o','s','^','D','X']
fig, axes = plt.subplots(len(class_names), 2, figsize=(18, 5*len(class_names)), sharey=True)
fig.suptitle(
    'B2-02 | Per-Rock Recall vs. Test-Split Ratio -- One Line per Data Fraction\n'
    'Each panel = one rock type on one belt speed\n'
    'A steeply falling line = that rock is most affected by fewer training images',
    fontsize=11, fontweight='bold'
)
for ci, cls in enumerate(class_names):
    for si, tag in enumerate(['1.83 Hz', '5.10 Hz']):
        ax = axes[ci][si]
        mat = recall_matrix(sweep_results[tag], cls)
        for fi, (frac, col, m) in enumerate(zip(DATA_FRACTIONS, lc, mk)):
            row = mat[fi]; valid = ~np.isnan(row)
            ax.plot(np.array(TEST_SIZES)[valid]*100, row[valid],
                    marker=m, color=col, ms=6, label=f'{int(frac*100)}% data')
        ax.set_title(f'{cls.replace("_"," ")} -- {tag}', fontsize=10)
        ax.set_xlabel('Test-Set Size (%)'); ax.set_ylabel('Recall (%)')
        ax.set_xticks([int(t*100) for t in TEST_SIZES])
        ax.set_ylim(-5, 107)
        ax.axhline(33.3, color='grey', ls='--', lw=0.7, alpha=0.4)
        ax.legend(title='Data fraction', fontsize=7, loc='lower left')
        ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, DIR_B2,
    'B2-02_perrock_recall_vs_testsplit__one_line_per_datafraction.png',
    'Block 2 | Per-rock recall curves vs. test-split ratio -- one line per data-fraction level (all rocks, both datasets)')
plt.show()

In [ ]:
# B2-03  Per-rock recall vs. data fraction  (one line per test-split)

lc2 = plt.cm.plasma(np.linspace(0.1, 0.9, len(TEST_SIZES)))
mk2 = ['o','s','^','D','v','<','>','p','*']
fig, axes = plt.subplots(len(class_names), 2, figsize=(18, 5*len(class_names)), sharey=True)
fig.suptitle(
    'B2-03 | Per-Rock Recall vs. Data Fraction -- One Line per Test-Split Size\n'
    'Each panel = one rock type on one belt speed\n'
    'A rising line = recall for that rock improves as more training images are available',
    fontsize=11, fontweight='bold'
)
for ci, cls in enumerate(class_names):
    for si, tag in enumerate(['1.83 Hz', '5.10 Hz']):
        ax = axes[ci][si]
        mat = recall_matrix(sweep_results[tag], cls)
        for ti, (ts, col, m) in enumerate(zip(TEST_SIZES, lc2, mk2)):
            col_d = mat[:,ti]; valid = ~np.isnan(col_d)
            ax.plot(np.array(DATA_FRACTIONS)[valid]*100, col_d[valid],
                    marker=m, color=col, ms=6, label=f'{int(ts*100)}% test')
        ax.set_title(f'{cls.replace("_"," ")} -- {tag}', fontsize=10)
        ax.set_xlabel('Data Fraction (%)'); ax.set_ylabel('Recall (%)')
        ax.set_xticks([int(f*100) for f in DATA_FRACTIONS])
        ax.set_ylim(-5, 107)
        ax.axhline(33.3, color='grey', ls='--', lw=0.7, alpha=0.4)
        ax.legend(title='Test-split', fontsize=7, loc='lower right')
        ax.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, DIR_B2,
    'B2-03_perrock_recall_vs_datafraction__one_line_per_testsplit.png',
    'Block 2 | Per-rock recall curves vs. data fraction -- one line per test-split size (all rocks, both datasets)')
plt.show()

In [ ]:
# B2-04  Normalised confusion matrix + per-rock recall side panel: BEST config

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle(
    'B2-04 | Normalised Confusion Matrix + Per-Rock Recall -- Best Configuration\n'
    'Each row sums to 100% (row = actual class)  |  Diagonal value = recall for that rock\n'
    'Cell shows raw count and row-normalised %  |  Side box = exact recall per rock',
    fontsize=11, fontweight='bold'
)
for ax, tag in zip(axes, ['1.83 Hz', '5.10 Hz']):
    vd = valid_dict(sweep_results[tag])
    bk = max(vd, key=lambda k: vd[k]['best_acc'])
    r = vd[bk]; frac, ts = bk
    cm_raw = r['cm'].astype(float)
    rs = cm_raw.sum(axis=1, keepdims=True)
    cm_norm = np.zeros_like(cm_raw)  
    np.divide(cm_raw, rs, out=cm_norm, where=rs != 0)
    annot = np.empty_like(r['cm'], dtype=object)
    for i in range(r['cm'].shape[0]):
        for j in range(r['cm'].shape[1]):
            annot[i,j] = f"{r['cm'][i,j]}\n({cm_norm[i,j]*100:.1f}%)"
    sns.heatmap(cm_norm, ax=ax, annot=annot, fmt='', cmap='Blues',
                xticklabels=short_names, yticklabels=short_names, vmin=0, vmax=1)
    rc_txt = 'Per-Rock Recall\n' + '-'*16 + '\n' + '\n'.join(
        f"{c.split('_')[0]}: {r['per_class_recall'][c]*100:.1f}%" for c in class_names)
    ax.text(1.37, 0.5, rc_txt, transform=ax.transAxes, va='center', ha='left',
            fontsize=9, family='monospace',
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))
    ax.set_title(f'Best Config -- {tag}\ndata={int(frac*100)}%  test-split={int(ts*100)}%  overall acc={r["best_acc"]*100:.1f}%', fontsize=10)
    ax.set_ylabel('Actual class'); ax.set_xlabel('Predicted class')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
save_fig(fig, DIR_B2,
    'B2-04_normalised_CM_with_perrock_recall__best_config_per_dataset.png',
    'Block 2 | Row-normalised CM (count + %) with per-rock recall side panel -- best config on each dataset')
plt.show()

In [ ]:
# B2-05  Radar chart: per-rock recall for BEST / MEDIAN / WORST config

def radar(ax, values_list, labels, series_labels, colors, title):
    N = len(labels)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
    angles += angles[:1]
    ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([l.replace('_','\n') for l in labels], fontsize=9)
    ax.set_ylim(0,100); ax.set_yticks([20,40,60,80,100])
    ax.set_yticklabels(['20','40','60','80','100'], fontsize=7, alpha=0.5)
    ax.set_title(title, fontsize=10, pad=18)
    for vals, slbl, clr in zip(values_list, series_labels, colors):
        v = vals + vals[:1]
        ax.plot(angles, v, color=clr, lw=2, label=slbl)
        ax.fill(angles, v, color=clr, alpha=0.08)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35,1.15), fontsize=7)

fig = plt.figure(figsize=(16, 7))
fig.suptitle(
    'B2-05 | Per-Rock Recall Radar -- Best / Median / Worst Configuration\n'
    'Each axis = one rock type  |  Distance from centre = recall %  |  Wider polygon = better per-rock recall',
    fontsize=11, fontweight='bold'
)
for pi, tag in enumerate(['1.83 Hz', '5.10 Hz']):
    ax = fig.add_subplot(1, 2, pi+1, polar=True)
    vd  = valid_dict(sweep_results[tag])
    srt = sorted(vd.items(), key=lambda x: x[1]['best_acc'])
    wk, wr = srt[0]; mk, mr = srt[len(srt)//2]; bk, br = srt[-1]
    get_rc = lambda r: [r['per_class_recall'][c]*100 for c in class_names]
    radar(ax, [get_rc(br), get_rc(mr), get_rc(wr)], class_names,
          [f'Best   {br["best_acc"]*100:.1f}% (d={int(bk[0]*100)}%,t={int(bk[1]*100)}%)',
           f'Median {mr["best_acc"]*100:.1f}% (d={int(mk[0]*100)}%,t={int(mk[1]*100)}%)',
           f'Worst  {wr["best_acc"]*100:.1f}% (d={int(wk[0]*100)}%,t={int(wk[1]*100)}%)'],
          ['#2ecc71','#3498db','#e74c3c'], title=tag)
plt.tight_layout()
save_fig(fig, DIR_B2,
    'B2-05_radar_perrock_recall__best_median_worst_config.png',
    'Block 2 | Radar chart comparing per-rock recall for best, median, and worst configurations on each dataset')
plt.show()

In [ ]:
# FIN-01  Full comparison heatmaps: overall acc + all per-rock recalls

for tag in ['1.83 Hz', '5.10 Hz']:
    tag_fs = tag.replace(' ','').replace('.','')
    n_cols = 1 + len(class_names)
    fig, axes = plt.subplots(1, n_cols, figsize=(7*n_cols, 6))
    fig.suptitle(
        f'FIN-01 | Overall Accuracy + Per-Rock Recall Heatmaps  [{tag}]\n'
        f'Leftmost panel = overall accuracy  |  Remaining panels = recall per rock type\n'
        f'Row = data fraction  |  Col = test-set size  |  All values in %',
        fontsize=11, fontweight='bold'
    )
    rd = sweep_results[tag]
    sns.heatmap(acc_matrix(rd), ax=axes[0],
                xticklabels=xl, yticklabels=yl,
                annot=True, fmt='.1f', cmap='RdYlGn', vmin=33, vmax=100,
                linewidths=0.5, linecolor='white', cbar_kws={'label': 'Accuracy (%)'})
    axes[0].set_title('Overall\nAccuracy (%)', fontsize=11, fontweight='bold')
    axes[0].set_xlabel('Test-Set Size'); axes[0].set_ylabel('Data Fraction')
    axes[0].tick_params(axis='x', rotation=0)
    for ci, cls in enumerate(class_names):
        ax = axes[ci+1]
        sns.heatmap(recall_matrix(rd, cls), ax=ax,
                    xticklabels=xl, yticklabels=yl,
                    annot=True, fmt='.1f', cmap='RdYlGn', vmin=0, vmax=100,
                    linewidths=0.5, linecolor='white', cbar_kws={'label': 'Recall (%)'})
        ax.set_title(f'{cls.replace("_"," ")}\nRecall (%)', fontsize=10)
        ax.set_xlabel('Test-Set Size'); ax.set_ylabel('')
        ax.tick_params(axis='x', rotation=0)
    plt.tight_layout()
    save_fig(fig, DIR_FIN,
        f'FIN-01_full_heatmaps__overall_accuracy_and_perrock_recall__{tag_fs}.png',
        f'Final | Overall accuracy + per-rock recall heatmaps combined in one figure  [{tag}]')
    plt.show()

In [ ]:
# FIN-02  Accuracy difference heatmap: 1.83 Hz minus 5.10 Hz

diff = acc_matrix(sweep_results['1.83 Hz']) - acc_matrix(sweep_results['5.10 Hz'])
fig, ax = plt.subplots(figsize=(11, 6))
fig.suptitle(
    'FIN-02 | Accuracy Difference Heatmap:  1.83 Hz  minus  5.10 Hz  (percentage points)\n'
    'Red = 1.83 Hz gives higher accuracy  |  Blue = 5.10 Hz gives higher accuracy  |  White = equal\n'
    'Row = data fraction  |  Col = test-set size',
    fontsize=11, fontweight='bold'
)
sns.heatmap(diff, ax=ax, xticklabels=xl, yticklabels=yl,
            annot=True, fmt='.1f', cmap='coolwarm', center=0,
            linewidths=0.5, linecolor='white', cbar_kws={'label': 'Accuracy difference (pp)'})
ax.set_xlabel('Test-Set Size'); ax.set_ylabel('Data Fraction')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
save_fig(fig, DIR_FIN,
    'FIN-02_accuracy_difference_heatmap__1.83Hz_minus_5.10Hz.png',
    'Final | Accuracy difference (pp) heatmap: 1.83 Hz minus 5.10 Hz across the full sweep grid')
plt.show()

In [ ]:
# FIN-03  Summary bar charts: overall accuracy range + mean per-rock recall

summary = {}
for tag in ['1.83 Hz', '5.10 Hz']:
    vd   = valid_dict(sweep_results[tag])
    accs = [v['best_acc']*100 for v in vd.values()]
    summary[tag] = {'max': np.nanmax(accs), 'mean': np.nanmean(accs),
                    'min': np.nanmin(accs), 'std': np.nanstd(accs)}
    for cls in class_names:
        rc = [v['per_class_recall'][cls]*100 for v in vd.values()]
        summary[tag][f'mean_recall_{cls}'] = np.nanmean(rc)

tags = ['1.83 Hz', '5.10 Hz']
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    'FIN-03 | Summary Bar Charts -- Accuracy Range and Mean Per-Rock Recall\n'
    'Left: max/mean/min overall accuracy across the entire sweep grid per dataset\n'
    'Right: mean recall for each rock averaged over all (data-fraction, test-split) combinations',
    fontsize=11, fontweight='bold'
)
ax1 = axes[0]; xp = np.arange(len(tags)); bw = 0.25
for i, (vals, clr, lbl) in enumerate([
    ([summary[t]['max']  for t in tags], '#2ecc71', 'Max acc'),
    ([summary[t]['mean'] for t in tags], '#3498db', 'Mean acc'),
    ([summary[t]['min']  for t in tags], '#e74c3c', 'Min acc'),
]):
    xs = xp + (i-1)*bw
    bars = ax1.bar(xs, vals, bw, color=clr, label=lbl, zorder=3)
    for b, v in zip(bars, vals):
        ax1.text(b.get_x()+b.get_width()/2, v+0.5, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
ax1.set_xticks(xp); ax1.set_xticklabels(tags)
ax1.set_ylabel('Accuracy (%)'); ax1.set_title('Overall Accuracy Range Across All Configurations')
ax1.set_ylim(0,115); ax1.legend(); ax1.grid(True, alpha=0.3, axis='y', zorder=0)

ax2 = axes[1]; x2 = np.arange(len(class_names))
for si, (tag, clr) in enumerate(zip(tags, ['#9b59b6','#f39c12'])):
    vals   = [summary[tag][f'mean_recall_{c}'] for c in class_names]
    offset = (si-0.5)*0.4
    bars   = ax2.bar(x2+offset, vals, 0.4, label=tag, color=clr, zorder=3)
    for b, v in zip(bars, vals):
        ax2.text(b.get_x()+b.get_width()/2, v+0.5, f'{v:.1f}', ha='center', va='bottom', fontsize=8)
ax2.set_xticks(x2); ax2.set_xticklabels(short_names)
ax2.set_ylabel('Mean Recall (%)'); ax2.set_title('Mean Per-Rock Recall Averaged Over Entire Sweep Grid')
ax2.set_ylim(0,115); ax2.legend(); ax2.grid(True, alpha=0.3, axis='y', zorder=0)

plt.tight_layout()
save_fig(fig, DIR_FIN,
    'FIN-03_summary_bars__accuracy_range_and_mean_perrock_recall.png',
    'Final | Summary bars: overall accuracy range (max/mean/min) and mean per-rock recall across the full sweep')
plt.show()

In [ ]:
# FIN-04  Best & worst normalised confusion matrices: all 4 combinations

fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle(
    'FIN-04 | Normalised Confusion Matrices -- Best & Worst Config on Each Dataset\n'
    'Top row = 1.83 Hz  |  Bottom row = 5.10 Hz  |  Left = BEST config  |  Right = WORST config\n'
    'Each cell: raw count + row % (row = actual class, rows sum to 100%)',
    fontsize=11, fontweight='bold'
)
for ri, tag in enumerate(['1.83 Hz', '5.10 Hz']):
    vd = valid_dict(sweep_results[tag])
    bk = max(vd, key=lambda k: vd[k]['best_acc'])
    wk = min(vd, key=lambda k: vd[k]['best_acc'])
    for ci, (key, lbl, cmap) in enumerate([(bk,'BEST','Blues'),(wk,'WORST','Reds')]):
        r = vd[key]; ax = axes[ri][ci]; frac, ts = key
        cm_raw = r['cm'].astype(float)
        rs = cm_raw.sum(axis=1, keepdims=True)
        cm_norm = np.zeros_like(cm_raw)  
        np.divide(cm_raw, rs, out=cm_norm, where=rs != 0)
        annot = np.empty_like(r['cm'], dtype=object)
        for i in range(r['cm'].shape[0]):
            for j in range(r['cm'].shape[1]):
                annot[i,j] = f"{r['cm'][i,j]}\n{cm_norm[i,j]*100:.1f}%"
        sns.heatmap(cm_norm, ax=ax, annot=annot, fmt='', cmap=cmap,
                    xticklabels=short_names, yticklabels=short_names, vmin=0, vmax=1)
        ax.set_title(
            f'[{lbl}] {tag}\n'
            f'data={int(frac*100)}%  test-split={int(ts*100)}%  '
            f'overall acc={r["best_acc"]*100:.1f}%  '
            f'n_train={r["n_train"]}  n_test={r["n_test"]}', fontsize=9)
        ax.set_ylabel('Actual class'); ax.set_xlabel('Predicted class')
        ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
save_fig(fig, DIR_FIN,
    'FIN-04_normalised_CM__best_and_worst_configs_both_datasets.png',
    'Final | 2x2 normalised CMs (count + row%): best vs. worst config on 1.83 Hz and 5.10 Hz')
plt.show()

In [ ]:
# FIN-05  Per-rock recall difference heatmaps: 1.83 Hz minus 5.10 Hz

fig, axes = plt.subplots(1, len(class_names), figsize=(8*len(class_names), 6))
fig.suptitle(
    'FIN-05 | Per-Rock Recall Difference:  1.83 Hz  minus  5.10 Hz  (percentage points)\n'
    'Each panel = one rock type  |  Red = 1.83 Hz gives higher recall for that rock\n'
    'Blue = 5.10 Hz gives higher recall  |  White = equal recall for that rock on both speeds',
    fontsize=11, fontweight='bold'
)
for ax, cls in zip(axes, class_names):
    diff_rc = (
        recall_matrix(sweep_results['1.83 Hz'], cls) -
        recall_matrix(sweep_results['5.10 Hz'], cls)
    )
    abs_max = np.nanmax(np.abs(diff_rc))
    sns.heatmap(diff_rc, ax=ax, xticklabels=xl, yticklabels=yl,
                annot=True, fmt='.1f', cmap='coolwarm', center=0,
                vmin=-abs_max, vmax=abs_max,
                linewidths=0.5, linecolor='white', cbar_kws={'label': 'Recall diff (pp)'})
    ax.set_title(f'{cls.replace("_"," ")}\nRecall diff (pp)', fontsize=11)
    ax.set_xlabel('Test-Set Size'); ax.set_ylabel('Data Fraction')
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
save_fig(fig, DIR_FIN,
    'FIN-05_perrock_recall_difference__1.83Hz_minus_5.10Hz.png',
    'Final | Per-rock recall difference (pp) between 1.83 Hz and 5.10 Hz -- one panel per rock, full sweep grid')
plt.show()

In [ ]:
# SAVE NUMERICAL RESULTS TO CSV
csv_path = os.path.join(RESULTS_DIR, 'results_all_runs.csv')
fieldnames = (
    ['dataset', 'data_fraction_pct', 'test_split_pct', 'n_train', 'n_test', 'best_acc_pct'] +
    [f'recall_{c}_pct' for c in class_names] +
    [f'f1_{c}' for c in class_names]
)
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=fieldnames)
    w.writeheader()
    for tag in ['1.83 Hz', '5.10 Hz']:
        for frac in DATA_FRACTIONS:
            for ts in TEST_SIZES:
                r = sweep_results[tag].get((frac, ts))
                row = {'dataset': tag,
                       'data_fraction_pct': int(frac*100),
                       'test_split_pct':    int(ts*100),
                       'n_train':      r['n_train']           if r else 'SKIP',
                       'n_test':       r['n_test']            if r else 'SKIP',
                       'best_acc_pct': round(r['best_acc']*100,2) if r else 'SKIP'}
                for c in class_names:
                    row[f'recall_{c}_pct'] = round(r['per_class_recall'][c]*100,2) if r else 'SKIP'
                    row[f'f1_{c}']         = round(r['per_class_f1'][c],4)         if r else 'SKIP'
                w.writerow(row)
_saved_files.append((csv_path,
    'All numerical results: one row per run -- columns = dataset, data-fraction, test-split, n_train, n_test, accuracy, per-rock recall and F1'))
print(f'[SAVED] {csv_path}')

In [ ]:
# WRITE RESULTS INDEX

index_path = os.path.join(RESULTS_DIR, 'RESULTS_INDEX.txt')

groups = {'block1_overall_sweep': [], 'block2_perrock_analysis': [],
          'final_comparison': [], '(root)': []}
for path, desc in _saved_files:
    placed = False
    for grp in ['block1_overall_sweep','block2_perrock_analysis','final_comparison']:
        if grp in path: groups[grp].append((path, desc)); placed = True; break
    if not placed: groups['(root)'].append((path, desc))

section_titles = {
    'block1_overall_sweep'   : 'BLOCK 1 -- Overall Accuracy Sweep',
    'block2_perrock_analysis': 'BLOCK 2 -- Per-Rock Recall Analysis',
    'final_comparison'       : 'FINAL COMPARISON',
    '(root)'                 : 'ROOT FILES (CSV + this index)',
}
with open(index_path, 'w') as f:
    f.write('RESULTS INDEX -- split_fraction_study\n')
    f.write('=' * 80 + '\n')
    f.write(f'EPOCHS={EPOCHS}  LR={LR}  WEIGHT_DECAY={WEIGHT_DECAY}  BATCH={BATCH_SIZE}\n')
    f.write(f'Data fractions tested : {DATA_FRACTIONS}\n')
    f.write(f'Test-split sizes tested: {TEST_SIZES}\n')
    f.write(f'Total training runs   : {len(DATA_FRACTIONS)*len(TEST_SIZES)*2}\n')
    f.write('=' * 80 + '\n\n')
    for grp, entries in groups.items():
        if not entries: continue
        f.write(section_titles[grp] + '\n')
        f.write('-' * 80 + '\n')
        for path, desc in entries:
            f.write(f'  {os.path.basename(path)}\n')
            f.write(f'    WHAT IT SHOWS: {desc}\n\n')
        f.write('\n')

print(f'[SAVED] {index_path}')
print(f'\nTotal files saved: {len(_saved_files)}')
print(f'\n  block1_overall_sweep/    : {sum(1 for p,_ in _saved_files if "block1" in p)} files')
print(f'  block2_perrock_analysis/ : {sum(1 for p,_ in _saved_files if "block2" in p)} files')
print(f'  final_comparison/        : {sum(1 for p,_ in _saved_files if "final_" in p)} files')
print(f'  root (csv + index)       : {sum(1 for p,_ in _saved_files if "block" not in p and "final" not in p)} files')
print('\n' + '='*50)
with open(index_path) as f:
    print(f.read())